In [257]:
import re
import json
from bs4 import BeautifulSoup

## Open the json and fill a dictionary

In [ ]:
# def open_monsters():
#     with open('./data/monsters_pretty.json', 'r', encoding="utf-8") as f_:
#         monsters_dict = json.load(f_)
#     return monsters_dict

def open_monsters():
    with open('./data/monsters_pretty.json', 'r', encoding="utf-8") as f_:
        monsters_dict = json.load(f_)
    return {k.replace('\u200e', '').strip(): v for k, v in monsters_dict.items()}

monsters_dict = open_monsters()

In [259]:

monster_text = monsters_dict["outsiders_demon_balor"]
print(monster_text)


<html>
 <head>
 </head>
 <body>
  <div class="article-content" id="article-content">
   <div class="breadcrumbs">
    <a class="bread-parent" href="https://www.d20pfsrd.com/" parentid="52278" title="Home">
     Home
    </a>
    &gt;
    <a class="bread-parent" href="https://www.d20pfsrd.com/bestiary/" parentid="57186" title="Bestiary">
     Bestiary
    </a>
    &gt;
    <a class="bread-parent" href="https://www.d20pfsrd.com/bestiary/monster-listings/" parentid="57428" title="(Bestiary) By Type">
     (Bestiary) By Type
    </a>
    &gt;
    <a class="bread-parent" href="https://www.d20pfsrd.com/bestiary/monster-listings/outsiders/" parentid="62127" title="Outsiders">
     Outsiders
    </a>
    &gt;
    <a class="bread-parent" href="https://www.d20pfsrd.com/bestiary/monster-listings/outsiders/demon/" parentid="62468" title="Demons">
     Demons
    </a>
    &gt;
   </div>
   <h1>
    Demon, Balor
   </h1>
   <script type="text/javascript">
    ognCreateVideoAdSpotOutstream("nitropay-

## remove top lines up to <h1>

In [260]:
top_patterns = [r'^.*?(?=<h1)',
                r'<script[^>]*>.*?</script>',
                r'<div class="toc_light_blue[^>]*>.*?</div>'
                ]

def scrub(monsters_dict, scrub_patterns):
    for monster in monsters_dict:
        for pattern in scrub_patterns:
            monsters_dict[monster] = re.sub(pattern, '', monsters_dict[monster], flags=re.DOTALL)

scrub(monsters_dict, top_patterns)



print(monsters_dict["outsiders_demon_balor"])

<h1>
    Demon, Balor
   </h1>
   
   
   <div class="ogn-childpages" style="float:right">
    <p>
     <b>
      Subpages
     </b>
    </p>
    <ul class="ogn-childpages">
     <li class="page new parent" post_id="62472">
      <a href="https://www.d20pfsrd.com/bestiary/monster-listings/outsiders/demon/balor/balor-lord/">
       Demon, Balor Lord
      </a>
     </li>
    </ul>
   </div>
   <div class="statblock">
    <p class="description">
     This winged fiend’s horned head and fanged visage present the perfection of the demonic form, fire spurting from its flesh.
    </p>
    <p class="title">
     Balor
     <span class="level">
      CR 20
     </span>
    </p>
    <p>
     <b>
      XP 307,200
     </b>
     <br/>
     CE Large
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#Outsider" rel="nofollow">
      outsider
     </a>
     (
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Chaotic" rel="nofollow">
 

## Check if it is a category or monster

In [261]:
def check_has_child(monsters_dict):
    has_child = []
    no_child = []
    for entry in monsters_dict:
        match = re.search(r'<div class="ogn-childpages"', monsters_dict[entry])
        if match:
            has_child.append(entry)
        else:
            no_child.append(entry)
    return has_child, no_child

has_child, no_child = check_has_child(monsters_dict)

print(f"{len(has_child)} {len(no_child)}")

436 5641


In [262]:
def check_if_monster(has_child):
    has_child_has_stats = []
    has_child_no_stats = []
    for entry in has_child:
        match = re.search(r'<p class="divider">\s*STATISTICS\s*</p>', monsters_dict[entry])
        # match = re.search(r'<div class="statblock">', monsters_dict[entry])
        if match:
            has_child_has_stats.append(entry)
        else:
            has_child_no_stats.append(entry)
            
    return has_child_has_stats, has_child_no_stats

has_child_has_stats, has_child_no_stats = check_if_monster(has_child)

# print(monsters_dict[has_child_no_stats[1]])
# print(f"{len(has_child_has_stats)} {len(has_child_no_stats)}")
print(f" no child {len(no_child)} \n has child, no stats {len(has_child_has_stats)} \n has child has stats {len(has_child_no_stats)}")


 no child 5641 
 has child, no stats 219 
 has child has stats 217


In [263]:
def split_dict(original_dict, split_list):
    split_dict = {}
    for entry in split_list:
        split_entry = original_dict.pop(entry)
        split_dict.update({entry: split_entry})
    print(f'original: {len(original_dict)}')
    print(f'split: {len(split_dict)}')
    return original_dict, split_dict

monsters_dict, categories_dict = split_dict(monsters_dict, has_child_no_stats)


original: 5860
split: 217


## check if monster is present twice

In [264]:
for monster in monsters_dict:
    match = re.search(r'<div class="ogn-childpages" style="float:right">.*?</div>', monsters_dict[monster], flags=re.DOTALL)    
    if match:
        pass
    name = monster.split("_")
    if name[-1] in name[:-1]:
        print(name)

['animals', 'shark', 'shark']
['animals', 'boar', 'boar']
['aberrations', 'ceroptor', 'ceroptor']
['animals', 'crocodilians', 'crocodile', 'crocodile']
['animals', 'elephant', 'elephant']
['animals', 'canines', 'dog', 'dog']
['animals', 'horse', 'horse']
['outsiders', 'kyton', 'kyton']
['animals', 'bat', 'bat']
['animals', 'cetacean', 'dolphin', 'dolphin']


## Remove Childpages Div

In [265]:
scrub_patterns = [r'<div class="ogn-childpages" style="float:right">.*?</div>', r'<div class="product-right">.*?</div>']

scrub(monsters_dict, scrub_patterns)
print(len(monsters_dict))

5860


## check if statblock

In [266]:
statblock_pattern = r'<div class="statblock">(.*?)</div>'
thirdpp_pattern = r'3pp'
description_pattern = r'<p class="description">.*?<p>'

def check_content(monsters_dict, pattern):
    yes = []
    no = []
    for monster in monsters_dict:
        monster_text = monsters_dict[monster]
        match = re.search(pattern, monster_text, flags=re.DOTALL | re.IGNORECASE)
        if match:
            yes.append(monster)
        else:
            no.append(monster)
    return yes, no


third_party, vanilla = check_content(monsters_dict, thirdpp_pattern)
print("3pp y/n:", len(third_party), len(vanilla))

yes_statblock, no_statblock = check_content(monsters_dict, statblock_pattern)
print("statblock y/n:", len(yes_statblock), len(no_statblock))

yes_description, no_description = check_content(monsters_dict, description_pattern)
print("description y/n:", len(yes_description), len(no_description))

3pp y/n: 494 5366
statblock y/n: 3215 2645
description y/n: 4006 1854


## try to Identify the Title/Header/Creature block

In [267]:
title_header_pattern = r'<p class="(?:title|header|creature)".*?(<span(?: class="(?:level|creature-level)")?)?.*?</p>'
h1_pattern = r'<h1>.*?</h1>'

yes, no = check_content(monsters_dict, h1_pattern)
print("h1 y/n:", len(yes), len(no))
yes_head, no_head = check_content(monsters_dict, title_header_pattern)
print("title/header y/n:", len(yes_head), len(no_head))
monsters_dict, monsters_table_dict = split_dict(monsters_dict, no_head)








h1 y/n: 5860 0
title/header y/n: 5747 113
original: 5747
split: 113


In [268]:
for monster in monsters_dict:
    match = re.search(r'<p class="header">\s*APPEARANCE\s*</p>', monsters_dict[monster], re.DOTALL)
    if match:
        print(monster)
        re.sub(r'<p class="header">\s*APPEARANCE\s*</p>', '', monsters_dict[monster], re.DOTALL)

monstrous-humanoids_hag-mute
plants_cauldron-bloom-2
fey_duende
fey_bulabar-2
outsiders_devil_devil-suffragan
monstrous-humanoids_hag-ash
fey_mythic-dryad
fey_elananx
fey_brownie-2
outsiders_dream-spectre-2
fey_fey-wolf
fey_cold-rider-2
fey_fear-eater-2
fey_blodeuwedd-2
outsiders_hag-dreamthief
fey_summer-hora-queen
fey_gremlin-erinat
fey_choxani-2
fey_mythic-fossegrim
undead_cyhyraeth
fey_mythic-gremlin-jinkin
fey_gremlin-hobkins
humanoids_angurboda
fey_hora-summer
fey_mythic-forlarren
fey_horzitoth
outsiders_daemons_daemon-venedaemon-2
fey_fey-courtier
fey_escorite-2
constructs_guardian-doll-2
fey_mythic-fastachee
fey_fey-grizzly-bear
aberrations_mythic-fachen
fey_draxie
fey_bogeyman-2
undead_danse-macabre-2
outsiders_azata-yamah
undead_death-coach-2
fey_buckawn-2
fey_mythic-gremlin-fuath
outsiders_daemons_daemon-harbinger-of-broken-deals-fine-print-and-unfair-bargains
fey_mythic-gremlin-nuglub
outsiders_devil_devil-joyful-thing
fey_crossroads-guardian
fey_buckawn-gang-leader
fey_gum

/tmp/ipykernel_97242/1711947061.py:5: DeprecationWarning: 'count' is passed as positional argument
  re.sub(r'<p class="header">\s*APPEARANCE\s*</p>', '', monsters_dict[monster], re.DOTALL)


## split off Mythic creatures

In [269]:
mythic_list = []

for monster in monsters_dict:
    head = re.search(r'<p class="(?:title|header|creature)".*?(<span(?: class="(?:level|creature-level)")?)?.*?</p>', monsters_dict[monster], re.DOTALL).group(0)
    if re.search(r'Mythic', head) and re.search(r'MR', head):
        mythic_list.append(monster)

monsters_dict, monsters_mythic_dict = split_dict(monsters_dict, mythic_list)

original: 5586
split: 161


## Summary

In [270]:
print(len(monsters_dict),len(monsters_mythic_dict),len(monsters_table_dict),len(categories_dict) )

5586 161 113 217


# Parsing

In [271]:
def clean_text(text):
    clean = re.sub(r'<[^>]+>', '', text).strip()
    clean = re.sub(r'\s+', ' ', clean).strip()
    return clean

#### save h1 and description

In [272]:
h1_pattern = r'<h1>(.*?)</h1>'

monsters_data = {}

for monster in monsters_dict:
    match = re.search(h1_pattern, monsters_dict[monster], re.DOTALL)
    title = clean_text(match.group(0))
    # print(title)
    monsters_data.update({monster: {"title1": title,
                                    "title2": monster.split("_")[-1]}})
    
    monsters_dict[monster] = re.sub(r'<h1>.*?</h1>', '', monsters_dict[monster], flags=re.DOTALL).strip()

print(len(monsters_data), len(monsters_dict))
# print(monsters_data["humanoids_hobgoblin-commander"])
for monster in monsters_data:
    print(monsters_data[monster]["title1"])

5586 5586
Gargoyle, Green Guardian (3pp)
Swan, Trumpeter
Golem, Magnesium (3pp)
Ettercap, Spider Lord
Clockwork Mage, Wrathplated
Pyrohydra (10-headed)
Jellyfish, Crimson
Letztermensch
Gohl
Rabisu
Walrus, Emperor
Redcap (3pp)
Hag, Annis
Demon, Schir
Lightning Cat Hero (3pp)
Boggard
Golem, Behemoth
Archon, Harbinger
Invisible Stalker, Advanced
Skeleton, Ogre
Vahana
Funglet
Alebrije
Dinosaur, Allosaurus
Clockwork Songbird
Clockwork Titan
Zoanoid, Gregole
Maelae Youth (Maelbud)
Herd Animal, Moose
Manasaputra, Rishi Manu
Damnation Book
Megafauna, Elasmotherium
Dream Eater (3pp)
Toad
Apocalypse Horse
Herd Animal, Goat
Lurker Above
Broken Soul Lillend
Dragonleaf Tree (3pp)
Grootslang
Basilisk, Crimson
Boneneedle, Lesser
Badger
Troop, Hobgoblin
Dark Creeper
Ankheg
Voonith
Inevitable, Valharut
Roofgarden
Nereid
Waspite
Avatar of Alkumuoto
Devil, Demoriel (Twice-Exiled Seductress; 3pp)
Chelicerae
Cockatrice, Crossbred
Gelatinous Cube
Nightmare Dragon, Adult
Octopus, Giant Lake
Nightshade, Night

In [184]:
for monster in monsters_dict:
    match = re.search(r'<p class="description">(.*?)</p>', monsters_dict[monster], flags=re.DOTALL)   
    if match:
        desc = match.group(1).strip()
        desc = clean_text(desc)
        if len(desc) < 1:
            desc = re.search(r'<p class="description">.*?</p>(.*?)</p>', monsters_dict[monster], flags=re.DOTALL).group(1)
            desc = clean_text(desc)
        if len(desc) > 0 and desc[0] == "X":
            monsters_data[monster]["types_"] = desc
        else:
            monsters_data[monster]["desc_short"] = desc
        monsters_dict[monster] = re.sub(r'<p class="description">.*?</p>', '', monsters_dict[monster], flags=re.DOTALL).strip()
    else:
        monsters_data.update({monster: {"desc_short": ""}})


#### save source and remove trail

In [185]:
for monster in monsters_dict:
    match = re.search(r'Copyright.*$', monsters_dict[monster], flags=re.DOTALL | re.IGNORECASE)
    if match:
        # print(match.group(0))
        match_href = re.search(r'<a(.*?)href=.*?>(.*?)</a>', match.group(0), flags=re.DOTALL)
        match_p = re.search(r'<p.*?>(.*?)</p>', match.group(0), flags=re.DOTALL)
        match_i = re.search(r'<i.*?>(.*?)</i>', match.group(0), flags=re.DOTALL)
        match_div = re.search(r'<div.*?>(.*?)</div>', match.group(0), flags=re.DOTALL)
        if match_href:
            monsters_data[monster]['source'] = match_href.group(1).strip()
        elif match_p:
            source = re.sub(r'<[^>]+>', '', match_p.group(1)).strip()
            source = re.sub(r'\s+', ' ', source).strip()
            # print(source)
        elif match_i:
            source = re.sub(r'<[^>]+>', '', match_i.group(1)).strip()
            source = re.sub(r'\s+', ' ', source).strip()
        elif match_div:
            source = re.sub(r'<[^>]+>', '', match_div.group(1)).strip()
            source = re.sub(r'\s+', ' ', source).strip()
            monsters_data[monster]['source'] = source
        else:
            monsters_data[monster]['source'] = ""   
    else:
        monsters_data[monster]['source'] = ""


        

In [186]:
for monster in monsters_dict:
    monsters_dict[monster] = re.sub(
        r'<!--div style="clear:both">.*$',
        '', 
        monsters_dict[monster], 
        flags=re.DOTALL
    )
    # print(repr(monsters_dict[monster][-500:]))
    monsters_dict[monster] = re.sub(
        r'<div class="section15">.*$',
        '', 
        monsters_dict[monster], 
        flags=re.DOTALL
    )
        
monster_text = monsters_dict["humanoids_hobgoblin-commander"]
print(monster_text)



<p class="header">
    Hobgoblin Commander
    <span class="level">
     CR 12
    </span>
   </p>
   <p>
    <b>
     XP
    </b>
    19,200
    <br/>
    <a href="https://www.d20pfsrd.com/races/other-races/featured-races/arg-hobgoblin">
     Hobgoblin
    </a>
    <a href="https://www.d20pfsrd.com/classes/alternate-classes/samurai">
     samurai
    </a>
    13
    <br/>
    LE Medium
    <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Humanoid">
     humanoid
    </a>
    (
    <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Goblinoid">
     goblinoid
    </a>
    )
    <br/>
    <b>
     Init
    </b>
    +4;
    <b>
     Senses
    </b>
    <a href="https://www.d20pfsrd.com/gamemastering/special-abilities#TOC-Darkvision">
     darkvision
    </a>
    60 ft.;
    <a href="https://www.d20pfsrd.com/skills/perception">
     Perception
    </a>
    +0
   </p>
   <p class="divider">
    DEFENSE
   </p>
   <p>
    <b>
    

#### Remove APPEARANCE in top_

In [187]:
def remove_APPEARANCE(monsters_dict):
    top_pattern = r'<p class="(?:title|header|creature)".*?(<span(?: class="(?:level|creature-level)")?)?.*?</p>'
    for monster in monsters_dict:
        match = re.search(top_pattern, monsters_dict[monster], flags=re.DOTALL)
        top_ = clean_text(match.group(0))
        if top_ == "APPEARANCE":
            print(clean_text(monsters_dict[monster][:240]))
            monsters_dict[monster] = monsters_dict[monster].replace(match.group(0), '', 1)
    return monsters_dict

monsters_dict = remove_APPEARANCE(monsters_dict)


APPEARANCE Mute Hag CR 11 XP 12,800 CE Medium <a href="https://www
APPEARANCE Cauldron bloom CR 15 XP 51,200 N Colossal <a href="http
APPEARANCE Duende CR 2 XP 600 CN Small <a href="https://www.d20pfs
APPEARANCE Bulabar CR 1 XP 400 LN Tiny <a href="https://www.d20pfs
APPEARANCE Devil, Suffragan Suffragan CR 5 XP 1,600
APPEARANCE Ash Hag CR 5 XP 1,600 LE Medium <a href="https://www.d2
APPEARANCE Mythic Dryad CR 4/MR 1 XP 1,200 CG Medium <a href="htt
APPEARANCE Elananx CR 6 XP 2,400 NE Medium <a href="https://www.d2
APPEARANCE Mythic Brownie CR 1/MR 1 XP 400 N Tiny <a href="https:/
APPEARANCE Dream Spectre CR 9 XP 6,400 NE Medium <a href="https://
APPEARANCE Fey Wolf CR 3 XP 800 CE Large <a href="https://www.d20p
APPEARANCE Cold Rider CR 8 XP 4,800 CE Medium <a href="https://www
APPEARANCE Fear Eater CR 5 XP 1,600 NE Medium <a href="https://www
APPEARANCE Blodeuwedd CR 6 XP 2,400 CN Medium <a href="https://www
APPEARANCE Dreamthief Hag CR 11 XP 12,800 NE Medium <a href="https
APPEARANCE 

In [188]:
top_pattern = r'<p class="(?:title|header|creature)".*?(<span(?: class="(?:level|creature-level)")?)?.*?</p>'
for monster in monsters_dict:
    match = re.search(top_pattern, monsters_dict[monster], flags=re.DOTALL)
    top_ = clean_text(match.group(0))
    monsters_data[monster]["top_"] = top_
    monsters_dict[monster] = monsters_dict[monster].replace(match.group(0), '', 1)
    

In [189]:
for monster in monsters_data:
    print(monsters_data[monster])

{'title1': 'Gargoyle, Green Guardian (3pp)', 'title2': 'gargoyle-green-guardian-tohc', 'desc_short': 'This winged humanoid creature is carved of a strange green stone with eyes rich black in color.', 'source': '', 'top_': 'Green Guardian CR 4'}
{'title1': 'Swan, Trumpeter', 'title2': 'swan-trumpeter', 'desc_short': 'This large waterfowl has a wide wingspan, all-white feathers, a black bill, and a long, curving neck.', 'source': '', 'top_': 'Trumpeter Swan CR 1/3'}
{'title1': 'Golem, Magnesium (3pp)', 'title2': 'golem-magnesium-tohc', 'desc_short': 'Silvery-white powder shakes loose from this construct as it stomps across the floor. Its face is a featureless plane except for two gouges where there should be eyes.', 'source': '', 'top_': 'Magnesium Golem CR 6'}
{'desc_short': '', 'source': '', 'top_': 'Spider Lord Ettercap CR 12'}
{'title1': 'Clockwork Mage, Wrathplated', 'title2': 'clockwork-mage-wrathplated', 'desc_short': 'Fire dances across the surface of this faceless construct and 

In [190]:
def clean_text(text):
    clean = re.sub(r'<[^>]+>', '', text).strip()
    clean = re.sub(r'\s+', ' ', clean).strip()
    return clean

def extract_section(monsters_dict, monsters_data):
    for monster in monsters_dict:
        types_pattern = r'^(.*?)(?=<p class="(?:divider|header)">\s*DEFENSES?\s*</p>)'
        types_pattern_2 = r'^(.*?)(?=<b>\s*DEFENSES?\s*</b>)'
        match = re.search(types_pattern, monsters_dict[monster], flags=re.DOTALL | re.IGNORECASE)
        match2 = re.search(types_pattern_2, monsters_dict[monster], flags=re.DOTALL | re.IGNORECASE)
        if match:
            clean = clean_text(match.group(0))
            # print(clean)
        elif match2:
            clean = clean_text(match2.group(0))
            # print(clean)
        else:
            print(monster)

def extract_types(text, monster):
    if len(text) == 0:
        print(monster)


extract_section(monsters_dict, monsters_data)

print(monsters_dict["humanoids_gnoll"])
print(monsters_data["humanoids_gnoll"])

vermin_mantis-deadly
dragons_dragon_primal-brine
outsiders_angel-elpizo
dragons_dragon_dragon-cave-kp
dragons_dragon_dragon-jade
outsiders_scanderig-forgefiend
constructs_sentinel-bronze
outsiders_shadow-of-a-doubt
dragons_dragon_dungeon-dragon-tohc
animals_fish_fish-tiger
aberrations_pygmy-watcher
dragons_dragon_imperial-underworld
outsiders_caulborn_caulborn-chrestomath
dragons_dragon_imperial-sea
vermin_urchin_sea-urchin-black-spot
outsiders_lurker-in-darkness
undead_petrified-maiden
outsiders_genie_marid_shahzada-noble-marid
outsiders_angel-guiding-light
dragons_dragon_river-dragon
vermin_urchin_sea-urchin-ravenous-swarm
dragons_dragon_primal-magma
dragons_dream-imp-companion
plants_mold-russet
dragons_dragon_primal-crystal
outsiders_scarlet-walker
aberrations_moit-of-shub-niggurath_slugspawn
dragons_dragon_imperial-forest
vermin_urchin_sea-urchin-great-diadem

   
   </div>
   
   
   <p>
    <b>
     XP 400
    </b>
    <br/>
    CE Medium
    <a href="https://www.d20pfsrd.com/be

In [191]:
monster_text = monsters_dict["outsiders_demon_balor"]
print(monsters_data["outsiders_demon_balor"])
print(monster_text)

{'title1': 'Demon, Balor', 'title2': 'balor', 'desc_short': 'This winged fiend’s horned head and fanged visage present the perfection of the demonic form, fire spurting from its flesh.', 'source': '', 'top_': 'Balor CR 20'}
<div class="statblock">
    
    
    <p>
     <b>
      XP 307,200
     </b>
     <br/>
     CE Large
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#Outsider" rel="nofollow">
      outsider
     </a>
     (
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Chaotic" rel="nofollow">
      chaotic
     </a>
     ,
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Demon" rel="nofollow">
      demon
     </a>
     ,
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Evil" rel="nofollow">
      evil
     </a>
     ,
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#extraplanar" rel="nofollow">
      